# Assignment 3 - Clustering

This notebook compares clustering choices and writes `clusters.csv`.

The work is structured as: analysis first, action second. The final decision is
not based on a single score; it combines cosine silhouette, cluster sizes, top
terms, and representative documents.


## Run 1 - Analysis / rationale

Import libraries, define shared preprocessing, and load the data. The cleaning
function is repeated here so the notebook is self-contained and can be run
without depending on the exploration notebook.


In [1]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
if not (ROOT / "articles.csv").exists() and (ROOT.parent / "articles.csv").exists():
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 160)

def clean_text_for_clustering(text: str) -> str:
    """Remove artifacts that are not topical content.

    The goal is not to erase the document meaning. The goal is to stop the model
    from clustering on Usenet signatures, URLs, e-mail addresses, and random IDs.
    """
    text = str(text)

    # Common Usenet signature separator. Apply only if the separator is late
    # enough to plausibly be a signature, not a normal sentence break.
    matches = list(re.finditer(r"\s--\s", text))
    if matches:
        last = matches[-1]
        if last.start() > len(text) * 0.45 or len(text) - last.start() < 900:
            text = text[: last.start()]

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " ", text)
    text = re.sub(r"\b[A-Z0-9]{5,}(?:-[A-Z0-9]{2,})+\b", " ", text)
    text = re.sub(r"[_=]{2,}|-{3,}", " ", text)
    return text


def build_unsafe_features(texts: pd.Series) -> pd.DataFrame:
    """Features aimed at unsafe/corrupted marketplace-like documents.

    These are intentionally different from topic-clustering features. A document
    can be anomalous because of structure and repeated machine-like tokens, even
    if its individual words are common.
    """
    rows = []
    for text in texts.astype(str):
        urls = len(re.findall(r"https?://\S+|www\.\S+", text))
        listing = len(re.findall(r"LISTING_ID_", text))
        ecommerce = len(
            re.findall(
                r"\b(?:SKU|TOKEN|COUPON|PROMO(?:_ID|_REF)?|BUNDLE_REF|ORDER_REF|SELLER_CODE|PRICE_REF|SALE_CODE)\b",
                text,
            )
        )
        contact_market = len(
            re.findall(
                r"Contact:|\+32\s?\d|asking\s+\d+|euro|EUR|pickup|discount|coupon|offer|sale|cash only|quick pickup|fast deal|special discount",
                text,
                flags=re.I,
            )
        )
        bad_domains = len(
            re.findall(
                r"pickup-offer-fast|trusted-sales-direct|bonus-marketplace|clearance-deals|claim-discount-now|deal-bundle-now",
                text,
                flags=re.I,
            )
        )
        html_ui = len(
            re.findall(
                r"<(?:div|span|button|input|a)\b|</(?:div|span|button|a)>",
                text,
                flags=re.I,
            )
        )
        chars = list(text)
        char_len = len(text)
        digit_ratio = sum(c.isdigit() for c in chars) / max(char_len, 1)
        upper_ratio = sum(c.isupper() for c in chars) / max(char_len, 1)
        punct_ratio = sum((not c.isalnum() and not c.isspace()) for c in chars) / max(char_len, 1)
        words = re.findall(r"[A-Za-z]+", text)
        unique_word_ratio = len(set(w.lower() for w in words)) / max(len(words), 1)

        unsafe_score = (
            10 * listing
            + 3 * urls
            + 2 * ecommerce
            + contact_market
            + 4 * bad_domains
            + html_ui
        )
        rows.append(
            {
                "char_len": char_len,
                "word_count": len(words),
                "unique_word_ratio": unique_word_ratio,
                "digit_ratio": digit_ratio,
                "upper_ratio": upper_ratio,
                "punct_ratio": punct_ratio,
                "url_count": urls,
                "listing_marker_count": listing,
                "ecommerce_token_count": ecommerce,
                "market_phrase_count": contact_market,
                "bad_domain_count": bad_domains,
                "html_ui_token_count": html_ui,
                "unsafe_score": unsafe_score,
            }
        )
    return pd.DataFrame(rows)

from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, RobustScaler

articles = pd.read_csv(ROOT / "articles.csv")
clusters_template = pd.read_csv(ROOT / "clusters.csv")
assert clusters_template["doc_id"].tolist() == articles["doc_id"].tolist()

custom_stop = list(
    ENGLISH_STOP_WORDS.union(
        {
            "edu",
            "com",
            "article",
            "writes",
            "posting",
            "host",
            "nntp",
            "university",
            "subject",
            "lines",
            "organization",
        }
    )
)


## Run 2 - Analysis / rationale

The assignment says 50 unsafe files were mixed into the collection. If those are
left inside topic clustering, a model can spend clusters on unsafe-format
variants instead of article categories. I therefore identify structural outliers
using only formatting/unsafe-site indicators, not hidden class labels.


In [2]:
unsafe_features = build_unsafe_features(articles["text"])
unsafe_feature_cols = [
    "url_count",
    "listing_marker_count",
    "ecommerce_token_count",
    "market_phrase_count",
    "bad_domain_count",
    "html_ui_token_count",
    "unsafe_score",
]

X_unsafe = RobustScaler().fit_transform(np.log1p(unsafe_features[unsafe_feature_cols]))
contamination = 50 / len(articles)

iso = IsolationForest(n_estimators=500, contamination=contamination, random_state=42)
iso.fit(X_unsafe)
unsafe_features["isolation_score"] = -iso.decision_function(X_unsafe)
structural_outlier_idx = unsafe_features["isolation_score"].nlargest(50).index.to_numpy()

normal_mask = np.ones(len(articles), dtype=bool)
normal_mask[structural_outlier_idx] = False

print("structural outliers selected:", len(structural_outlier_idx))
display(
    unsafe_features.iloc[structural_outlier_idx]
    .assign(doc_id=articles.iloc[structural_outlier_idx]["doc_id"].values)
    .sort_values("isolation_score", ascending=False)
    [["doc_id", *unsafe_feature_cols, "isolation_score"]]
    .head(12)
)


structural outliers selected: 50


,doc_id,url_count,listing_marker_count,ecommerce_token_count,market_phrase_count,bad_domain_count,html_ui_token_count,unsafe_score,isolation_score
415,DOC_00416,28,1,22,81,28,0,331,0.071515
145,DOC_00146,28,1,21,84,28,0,332,0.070522
1135,DOC_01136,29,1,13,84,29,0,323,0.068226
1927,DOC_01928,28,1,18,81,28,0,323,0.067786
1751,DOC_01752,23,1,22,71,23,0,286,0.067660
575,DOC_00576,28,1,13,84,28,0,316,0.067346
880,DOC_00881,28,1,14,72,28,0,306,0.066687
1584,DOC_01585,25,1,19,75,25,0,298,0.066671
450,DOC_00451,28,1,14,82,28,0,316,0.066578
1068,DOC_01069,26,1,16,75,26,0,299,0.064823


## Run 3 - Analysis / rationale

Metric choice matters. I use cosine silhouette for validation because TF-IDF
documents are sparse, nonnegative, and length-normalized. Cosine compares the
angle of term-weight vectors, so two documents can be similar even if one is
longer.

Why not Pearson correlation?

- Pearson subtracts each document's mean feature value. In sparse text vectors,
  the many zero entries dominate that mean-centered calculation.
- Absence of thousands of terms should not become strong negative evidence.
- Cosine is the standard similarity for TF-IDF and works naturally with L2
  normalization. For L2-normalized vectors, squared Euclidean distance equals
  `2 * (1 - cosine_similarity)`, so KMeans on normalized vectors is compatible
  with cosine-based interpretation.

Dataset-size decision: with 2164 documents, KMeans and agglomerative clustering
are both feasible. For much larger corpora, agglomerative and spectral methods
become less attractive because they scale poorly with pairwise relationships.


In [3]:
def evaluate_kmeans(matrix, name, k_values, metric="cosine", sample_size=None):
    rows = []
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=42, n_init=100, max_iter=500)
        labels = model.fit_predict(matrix)
        if metric == "cosine" and sample_size is not None:
            sil = silhouette_score(
                matrix,
                labels,
                metric="cosine",
                sample_size=min(sample_size, matrix.shape[0]),
                random_state=42,
            )
        else:
            sil = silhouette_score(matrix, labels, metric=metric)
        rows.append(
            {
                "representation": name,
                "algorithm": "KMeans",
                "k": k,
                "silhouette_cosine": sil,
                "inertia": getattr(model, "inertia_", np.nan),
                "cluster_sizes": tuple(np.bincount(labels)),
            }
        )
    return pd.DataFrame(rows)


def mean_top_terms(tfidf_matrix, labels, terms, top_n=14):
    rows = []
    for cluster_id in sorted(np.unique(labels)):
        idx = np.where(labels == cluster_id)[0]
        mean_weights = np.asarray(tfidf_matrix[idx].mean(axis=0)).ravel()
        top_idx = mean_weights.argsort()[-top_n:][::-1]
        rows.append(
            {
                "cluster": int(cluster_id),
                "size": int(len(idx)),
                "top_terms": ", ".join(terms[top_idx]),
            }
        )
    return pd.DataFrame(rows)


def representative_docs(docs, dense_matrix, labels, centers, n=4):
    rows = []
    for cluster_id in sorted(np.unique(labels)):
        idx = np.where(labels == cluster_id)[0]
        sims = cosine_similarity(dense_matrix[idx], centers[cluster_id].reshape(1, -1)).ravel()
        chosen = idx[np.argsort(sims)[-n:][::-1]]
        for rank, local_idx in enumerate(chosen, start=1):
            text = re.sub(r"\s+", " ", docs.iloc[local_idx]["text"][:260])
            rows.append(
                {
                    "cluster": int(cluster_id),
                    "rank": rank,
                    "doc_id": docs.iloc[local_idx]["doc_id"],
                    "excerpt": text,
                }
            )
    return pd.DataFrame(rows)


## Run 4 - Analysis / rationale

First compare simple full-data representations. This is a sanity check, not the
final model. If full-data clustering spends small clusters on unsafe-format
variants, that supports separating anomaly detection from topical clustering.


In [4]:
cleaned_all = articles["text"].map(clean_text_for_clustering)

count_vec = CountVectorizer(
    stop_words=custom_stop,
    min_df=4,
    max_df=0.55,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)
tfidf_vec_all = TfidfVectorizer(
    stop_words=custom_stop,
    min_df=4,
    max_df=0.55,
    sublinear_tf=True,
    norm="l2",
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)

X_count_all = count_vec.fit_transform(cleaned_all)
X_tfidf_all = tfidf_vec_all.fit_transform(cleaned_all)

full_eval = pd.concat(
    [
        evaluate_kmeans(X_count_all, "count_bow_all_docs", range(2, 8), sample_size=1500),
        evaluate_kmeans(X_tfidf_all, "tfidf_all_docs", range(2, 8), sample_size=1500),
    ],
    ignore_index=True,
)
full_eval.to_csv(OUTPUT_DIR / "full_data_kmeans_comparison.csv", index=False)
display(full_eval.sort_values(["representation", "k"]))


,representation,algorithm,k,silhouette_cosine,inertia,cluster_sizes
0,count_bow_all_docs,KMeans,2,0.023646,337618.284720,"(2147, 17)"
1,count_bow_all_docs,KMeans,3,0.020849,311730.612935,"(2131, 17, 16)"
2,count_bow_all_docs,KMeans,4,0.020849,293285.321676,"(2130, 1, 17, 16)"
3,count_bow_all_docs,KMeans,5,0.016180,281240.070544,"(2130, 16, 4, 13, 1)"
4,count_bow_all_docs,KMeans,6,0.016662,269895.298566,"(16, 1, 2130, 7, 2, 8)"
5,count_bow_all_docs,KMeans,7,0.016678,260871.874168,"(2129, 2, 1, 9, 6, 1, 16)"
6,tfidf_all_docs,KMeans,2,0.016307,2110.497581,"(2114, 50)"
7,tfidf_all_docs,KMeans,3,0.014614,2099.729823,"(50, 368, 1746)"
8,tfidf_all_docs,KMeans,4,0.016239,2091.515589,"(50, 1408, 346, 360)"
9,tfidf_all_docs,KMeans,5,0.017222,2085.187676,"(380, 1170, 264, 300, 50)"


## Run 5 - Analysis / rationale

For the final topic model I cluster only structurally normal documents, then
assign the 50 unsafe documents to a dedicated anomaly cluster in the submitted
`clusters.csv`. This preserves all document IDs while preventing unsafe-format
noise from consuming topic clusters.

I add TruncatedSVD/LSA before KMeans because sparse TF-IDF has many nearly
orthogonal dimensions. LSA compresses correlated terms into a lower-dimensional
semantic space. At this dataset size, 50 dimensions is small enough to be stable
and fast, while still retaining interpretable topic structure.


In [5]:
normal_docs = articles.loc[normal_mask].reset_index(drop=True)
normal_original_indices = np.where(normal_mask)[0]
cleaned_normal = normal_docs["text"].map(clean_text_for_clustering)

tfidf_vec = TfidfVectorizer(
    stop_words=custom_stop,
    min_df=4,
    max_df=0.55,
    sublinear_tf=True,
    norm="l2",
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)
X_tfidf = tfidf_vec.fit_transform(cleaned_normal)
terms = np.array(tfidf_vec.get_feature_names_out())

lsa = make_pipeline(TruncatedSVD(n_components=50, random_state=42), Normalizer(copy=False))
X_lsa = lsa.fit_transform(X_tfidf)
explained = lsa.named_steps["truncatedsvd"].explained_variance_ratio_.sum()
print("normal documents:", len(normal_docs))
print("tf-idf shape:", X_tfidf.shape)
print("LSA explained variance ratio:", round(float(explained), 4))


normal documents: 2114
tf-idf shape: (2114, 5292)
LSA explained variance ratio: 0.1119


## Run 6 - Analysis / rationale

Now compare KMeans, agglomerative clustering, and spectral clustering on the
same LSA representation. KMeans is expected to be stronger here because it
directly optimizes compact centroid-based groups. Agglomerative clustering is
useful as a second method because it checks whether the same broad topical
structure appears under a different clustering family. Spectral clustering is
included as an additional safeguard: it can capture non-convex graph structure,
but it is more expensive and less naturally interpretable for this sparse text
setting.


In [6]:
kmeans_lsa_eval = evaluate_kmeans(X_lsa, "tfidf_lsa50_normal_docs", range(2, 10), metric="cosine", sample_size=None)

agg_rows = []
for k in range(2, 10):
    agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = agg.fit_predict(X_lsa)
    agg_rows.append(
        {
            "representation": "tfidf_lsa50_normal_docs",
            "algorithm": "AgglomerativeWard",
            "k": k,
            "silhouette_cosine": silhouette_score(X_lsa, labels, metric="cosine"),
            "inertia": np.nan,
            "cluster_sizes": tuple(np.bincount(labels)),
        }
    )

spectral_rows = []
for k in [5, 6, 8]:
    spectral = SpectralClustering(
        n_clusters=k,
        affinity="nearest_neighbors",
        n_neighbors=20,
        assign_labels="kmeans",
        random_state=42,
        n_init=20,
    )
    labels = spectral.fit_predict(X_lsa)
    spectral_rows.append(
        {
            "representation": "tfidf_lsa50_normal_docs",
            "algorithm": "SpectralNN20",
            "k": k,
            "silhouette_cosine": silhouette_score(X_lsa, labels, metric="cosine"),
            "inertia": np.nan,
            "cluster_sizes": tuple(np.bincount(labels)),
        }
    )

model_eval = pd.concat([kmeans_lsa_eval, pd.DataFrame(agg_rows), pd.DataFrame(spectral_rows)], ignore_index=True)
model_eval.to_csv(OUTPUT_DIR / "clustering_model_comparison.csv", index=False)
display(model_eval.sort_values(["algorithm", "k"]))


,representation,algorithm,k,silhouette_cosine,inertia,cluster_sizes
8,tfidf_lsa50_normal_docs,AgglomerativeWard,2,0.083259,NaN,"(1753, 361)"
9,tfidf_lsa50_normal_docs,AgglomerativeWard,3,0.085984,NaN,"(1312, 361, 441)"
10,tfidf_lsa50_normal_docs,AgglomerativeWard,4,0.082627,NaN,"(1049, 361, 441, 263)"
11,tfidf_lsa50_normal_docs,AgglomerativeWard,5,0.088566,NaN,"(785, 361, 441, 263, 264)"
12,tfidf_lsa50_normal_docs,AgglomerativeWard,6,0.088785,NaN,"(267, 361, 441, 263, 264, 518)"
13,tfidf_lsa50_normal_docs,AgglomerativeWard,7,0.087336,NaN,"(361, 264, 441, 263, 232, 518, 35)"
14,tfidf_lsa50_normal_docs,AgglomerativeWard,8,0.080468,NaN,"(441, 264, 249, 263, 232, 518, 35, 112)"
15,tfidf_lsa50_normal_docs,AgglomerativeWard,9,0.077417,NaN,"(334, 264, 249, 263, 232, 518, 35, 112, 107)"
0,tfidf_lsa50_normal_docs,KMeans,2,0.086366,1742.177045,"(1728, 386)"
1,tfidf_lsa50_normal_docs,KMeans,3,0.098695,1681.839010,"(386, 471, 1257)"


## Run 7 - Analysis / rationale

Silhouette is useful, but it is not sufficient. I also check stability over
several random seeds. If a k value has slightly better silhouette but much lower
agreement between repeated runs, it is a weaker final choice for a defensible
submission.


In [7]:
from sklearn.metrics import adjusted_rand_score

stability_rows = []
for k in [5, 6, 8]:
    labels_by_seed = []
    silhouettes = []
    for seed in range(10):
        model = KMeans(n_clusters=k, random_state=seed, n_init=20, max_iter=500)
        labels = model.fit_predict(X_lsa)
        labels_by_seed.append(labels)
        silhouettes.append(silhouette_score(X_lsa, labels, metric="cosine"))

    ari_scores = [
        adjusted_rand_score(labels_by_seed[i], labels_by_seed[j])
        for i in range(len(labels_by_seed))
        for j in range(i + 1, len(labels_by_seed))
    ]
    stability_rows.append(
        {
            "k": k,
            "silhouette_mean": np.mean(silhouettes),
            "silhouette_std": np.std(silhouettes),
            "pairwise_ari_mean": np.mean(ari_scores),
            "pairwise_ari_min": np.min(ari_scores),
        }
    )

stability_df = pd.DataFrame(stability_rows)
stability_df.to_csv(OUTPUT_DIR / "kmeans_stability_check.csv", index=False)
display(stability_df)


,k,silhouette_mean,silhouette_std,pairwise_ari_mean,pairwise_ari_min
0,5,0.120085,0.000228,0.972439,0.953897
1,6,0.118905,0.000256,0.955570,0.916535
2,8,0.119169,0.003299,0.839312,0.754966


## Run 8 - Analysis / rationale

Selection decision: KMeans with 6 normal clusters is chosen. K=5 has a similar
score but merges medical content into a large general cluster. K=8 is competitive
on silhouette, but its repeated-run stability is weaker and it starts splitting
coherent topics into subtopics such as food-related medical posts and generic
information-request posts. K=6 gives the clearest balance: general discussion,
space, hockey, computer graphics, medical/health, and autos.


In [8]:
SELECTED_NORMAL_K = 6
final_kmeans = KMeans(n_clusters=SELECTED_NORMAL_K, random_state=42, n_init=100, max_iter=500)
normal_raw_labels = final_kmeans.fit_predict(X_lsa)

raw_top_terms = mean_top_terms(X_tfidf, normal_raw_labels, terms, top_n=18)
display(raw_top_terms)


,cluster,size,top_terms
0,0,487,"just, like, don, think, people, know, does, say, way, right, did, good, make, question, want, things, real, time"
1,1,254,"space, orbit, moon, launch, earth, lunar, nasa, spacecraft, cost, think, station, like, idea, solar, shuttle, new, need, long"
2,2,378,"game, team, hockey, play, players, games, season, nhl, teams, win, year, think, league, playoffs, detroit, cup, toronto, player"
3,3,410,"thanks, graphics, program, files, know, help, looking, image, file, does, mail, hi, windows, software, information, using, advance, use"
4,4,315,"disease, msg, don, doctor, pain, patients, cause, blood, food, know, people, treatment, time, medicine, medical, diet, patient, like"
5,5,270,"car, cars, engine, dealer, price, like, new, just, ford, good, don, speed, drive, time, toyota, know, got, problem"


## Run 9 - Analysis / rationale

KMeans cluster numbers are arbitrary. For readability and reproducibility of the
submitted CSV, I remap raw cluster IDs to stable semantic integers:

0 = general discussion, 1 = space, 2 = hockey, 3 = computer graphics/software,
4 = medical/health, 5 = autos, 6 = unsafe listing/anomaly.

The remapping uses top terms only; it does not use hidden labels.


In [9]:
semantic_keywords = {
    1: {"space", "orbit", "moon", "launch", "nasa", "lunar", "spacecraft"},
    2: {"hockey", "nhl", "team", "players", "games", "playoffs", "season"},
    3: {"graphics", "image", "files", "program", "windows", "software", "gif"},
    4: {"disease", "doctor", "patients", "medical", "medicine", "blood", "pain", "treatment", "msg"},
    5: {"car", "cars", "engine", "dealer", "ford", "toyota", "drive", "speed"},
}

def infer_semantic_label(top_terms_text):
    terms_set = set(re.findall(r"[a-zA-Z]+", top_terms_text.lower()))
    scores = {
        semantic_id: len(terms_set.intersection(keywords))
        for semantic_id, keywords in semantic_keywords.items()
    }
    best_semantic, best_score = max(scores.items(), key=lambda kv: kv[1])
    return best_semantic if best_score > 0 else 0

raw_to_semantic = {}
used = set()
for _, row in raw_top_terms.iterrows():
    semantic = infer_semantic_label(row["top_terms"])
    if semantic in used:
        semantic = 0
    raw_to_semantic[int(row["cluster"])] = int(semantic)
    used.add(int(semantic))

# Any cluster not matched by topic-specific terms is the general discussion cluster.
for raw_id in sorted(np.unique(normal_raw_labels)):
    raw_to_semantic.setdefault(int(raw_id), 0)

print("raw_to_semantic:", raw_to_semantic)

normal_semantic_labels = np.array([raw_to_semantic[int(x)] for x in normal_raw_labels])
final_labels = np.full(len(articles), 6, dtype=int)
final_labels[normal_original_indices] = normal_semantic_labels

clustered = clusters_template.copy()
clustered["label"] = final_labels
clustered.to_csv(ROOT / "clusters.csv", index=False)

cluster_summary = (
    pd.DataFrame({"doc_id": articles["doc_id"], "cluster": final_labels})
    .groupby("cluster")
    .agg(size=("doc_id", "size"), examples=("doc_id", lambda s: list(s.head(5))))
    .reset_index()
)
display(cluster_summary)


raw_to_semantic: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}


,cluster,size,examples
0,0,487,"[DOC_00001, DOC_00007, DOC_00009, DOC_00015, DOC_00016]"
1,1,254,"[DOC_00002, DOC_00003, DOC_00038, DOC_00039, DOC_00044]"
2,2,378,"[DOC_00011, DOC_00024, DOC_00040, DOC_00043, DOC_00051]"
3,3,410,"[DOC_00005, DOC_00017, DOC_00020, DOC_00026, DOC_00033]"
4,4,315,"[DOC_00004, DOC_00008, DOC_00010, DOC_00029, DOC_00031]"
5,5,270,"[DOC_00006, DOC_00013, DOC_00014, DOC_00018, DOC_00022]"
6,6,50,"[DOC_00012, DOC_00081, DOC_00105, DOC_00146, DOC_00211]"


## Run 10 - Analysis / rationale

Top terms and representative documents are the qualitative check required by
the assignment. They tell us whether the numeric clusters correspond to
interpretable categories instead of arbitrary partitions.


In [10]:
semantic_names = {
    0: "general_discussion",
    1: "space",
    2: "hockey",
    3: "computer_graphics_software",
    4: "medical_health",
    5: "autos",
    6: "unsafe_listing_anomaly",
}

normal_top_terms = mean_top_terms(X_tfidf, normal_semantic_labels, terms, top_n=18)
normal_top_terms["cluster_name"] = normal_top_terms["cluster"].map(semantic_names)

rep_raw = representative_docs(normal_docs, X_lsa, normal_raw_labels, final_kmeans.cluster_centers_, n=5)
rep_raw["cluster"] = rep_raw["cluster"].map(raw_to_semantic)
rep_raw["cluster_name"] = rep_raw["cluster"].map(semantic_names)

anomaly_examples = articles.iloc[structural_outlier_idx][["doc_id", "text"]].head(5).copy()
anomaly_examples["cluster"] = 6
anomaly_examples["cluster_name"] = semantic_names[6]
anomaly_examples["rank"] = range(1, len(anomaly_examples) + 1)
anomaly_examples["excerpt"] = anomaly_examples["text"].str.replace(r"\s+", " ", regex=True).str[:260]
anomaly_examples = anomaly_examples[["cluster", "cluster_name", "rank", "doc_id", "excerpt"]]

top_terms_out = pd.concat(
    [
        normal_top_terms,
        pd.DataFrame(
            [
                {
                    "cluster": 6,
                    "size": int((final_labels == 6).sum()),
                    "top_terms": "listing, pickup, offer, coupon, token, discount, sale, contact, euro, url",
                    "cluster_name": semantic_names[6],
                }
            ]
        ),
    ],
    ignore_index=True,
)
representatives_out = pd.concat([rep_raw, anomaly_examples], ignore_index=True)

top_terms_out.to_csv(OUTPUT_DIR / "cluster_top_terms.csv", index=False)
representatives_out.to_csv(OUTPUT_DIR / "cluster_representatives.csv", index=False)

display(top_terms_out.sort_values("cluster"))
display(representatives_out.sort_values(["cluster", "rank"]).head(40))


,cluster,size,top_terms,cluster_name
0,0,487,"just, like, don, think, people, know, does, say, way, right, did, good, make, question, want, things, real, time",general_discussion
1,1,254,"space, orbit, moon, launch, earth, lunar, nasa, spacecraft, cost, think, station, like, idea, solar, shuttle, new, need, long",space
2,2,378,"game, team, hockey, play, players, games, season, nhl, teams, win, year, think, league, playoffs, detroit, cup, toronto, player",hockey
3,3,410,"thanks, graphics, program, files, know, help, looking, image, file, does, mail, hi, windows, software, information, using, advance, use",computer_graphics_software
4,4,315,"disease, msg, don, doctor, pain, patients, cause, blood, food, know, people, treatment, time, medicine, medical, diet, patient, like",medical_health
5,5,270,"car, cars, engine, dealer, price, like, new, just, ford, good, don, speed, drive, time, toyota, know, got, problem",autos
6,6,50,"listing, pickup, offer, coupon, token, discount, sale, contact, euro, url",unsafe_listing_anomaly


,cluster,rank,doc_id,excerpt,cluster_name
0,0,1,DOC_01616,The fantasy was that he had found something of fundamental importance to one of the hot questions of the day ('77). He really had very l...,general_discussion
1,0,2,DOC_01478,"Subject: Re: Eugenics / ;Probably within 50 years, a new type of eugenics will be possible. ;Maybe even sooner. We are now mapping the h...",general_discussion
2,0,3,DOC_02056,"You totally forgot the original post that you posted Allen. In that post you stated that the ""wrap"" was on top of and in addition to any...",general_discussion
3,0,4,DOC_01503,"I'm not very impressed by the old so-called ""prospecting"" work from LPI, it has almost all been geared towards industrially silly proces...",general_discussion
4,0,5,DOC_01611,"Well, I guess I'm left wondering just who all the 'light fascists' think *they* are. Yes, I understand the issues. I don't even particul...",general_discussion
5,1,1,DOC_00609,"Brian Yamauchi asks: [Regarding orbital billboards...] Well, I had been collecting data for next edition of the Commercial Space News/Sp...",space
6,1,2,DOC_00814,SSF is up for redesign again. Let's do it right this time! Let's step back and consider the functionality we want: [1] microgravity/vacu...,space
7,1,3,DOC_01111,"I was at an interesting seminar at work (UK's R.A.L. Space Science Dept.) on this subject, specifically on a small-scale Solar Sail prop...",space
8,1,4,DOC_01498,"Two developments have brought these type of activities back to the forefront in 1993. First, in February, the Russians deployed a 20-m r...",space
9,1,5,DOC_01981,"For the purpose of a contest, I'd bet some things could be cut. Like fuel for re-entry, any kind of heat shielding, etc., etc. Even stil...",space


## Run 11 - Analysis / rationale

Final validation: `clusters.csv` must preserve document order, contain one
integer label per article, and use no more than 10 clusters.


In [11]:
written_clusters = pd.read_csv(ROOT / "clusters.csv")
assert written_clusters["doc_id"].tolist() == articles["doc_id"].tolist()
assert written_clusters["label"].notna().all()
assert pd.api.types.is_integer_dtype(written_clusters["label"])
assert written_clusters["label"].nunique() <= 10

selected_silhouette = silhouette_score(X_lsa, normal_raw_labels, metric="cosine")
print("Wrote:", ROOT / "clusters.csv")
print("Number of final clusters:", written_clusters["label"].nunique())
print("Selected normal-topic KMeans cosine silhouette:", round(float(selected_silhouette), 4))
display(written_clusters["label"].value_counts().sort_index())


Wrote: /Users/alperendavran/Desktop/data mining hw3/clusters.csv
Number of final clusters: 7
Selected normal-topic KMeans cosine silhouette: 0.1189


label
0    487
1    254
2    378
3    410
4    315
5    270
6     50
Name: count, dtype: int64